# VoxTool Real-Adapter Colab Demo

This notebook demonstrates the advanced benchmark adapters (Voxtral, Qwen,
Gemma) behind the shared `ModelAdapter` interface. It first walks through a
single adapter end to end (select, run text examples through Pipeline A,
generate test audio from text and run an audio pipeline, validate the canonical
JSON envelope, execute tools, show final answers and metrics), then **runs each
model over the same labeled, bilingual (English + Russian) data and compares
their metrics — per language, and across all four pipelines (A–D)**:

- **A** — text in → JSON tool-call envelope
- **B** — audio in → ASR transcript (WER vs the reference transcript)
- **C** — audio in → direct audio tool-call envelope
- **D** — audio in → ASR transcript then text-LLM tool-call envelope

The mock adapter requires no download and is ideal for a dry run; it recognizes
both English and Russian conversion phrasing. The real adapters (`qwen`,
`gemma`, `voxtral`) fetch weights lazily on first use and need GPU/Colab
resources.

Ordinary CI never runs this notebook against real models. Run real adapters
manually here in Colab.

## 1. Load the repository

Clone the repo in Colab and add it to `sys.path` so the packages import.

In [5]:
import os
import sys

REPO_URL = "https://github.com/nkz-soft/voxtool.git"
if not os.path.exists("voxtool"):
    !git clone -q $REPO_URL
sys.path.insert(0, os.path.abspath("voxtool"))

## 2. Install dependencies

Install the project plus the optional model extras. In Colab this pulls the
heavy `transformers`/`torch` stack used only for real adapters.

In [7]:
# Colab dependency install (skip when running a local checkout).
%pip install -q -e '.[model,notebook]'  # noqa

Note: you may need to restart the kernel to use updated packages.


D:\Projects\open-source\nz\voxtool\.venv\Scripts\python.exe: No module named pip


## 3. Select an adapter

List the registered adapters and build one from its config. No model is
downloaded at this step.

In [ ]:
from apps.notebook import colab_demo_helpers as demo

print("available adapters:", demo.list_adapters())
# Use 'mock' for a no-download dry run, or 'qwen'/'gemma'/'voxtral' for real.
ADAPTER_ID = "mock"
adapter = demo.select_adapter(ADAPTER_ID)
adapter.adapter_id, adapter.capabilities.model_dump()

## 4. Run text examples

Run a few prompts through Pipeline A. Each record preserves raw output,
parsed JSON, validation results, tool execution, and the final answer.

In [ ]:
prompts = [
    "Convert 2 kilometers to meters.",
    "What is the capital of France?",
    "Сконвертируй 5 километров в метры.",  # Russian conversion prompt
]
records = demo.run_text_demo(adapter, prompts, run_id="colab-demo")
[demo.record_summary(r) for r in records]

## 5. Generate test audio from text

Instead of uploading clips, synthesize deterministic test audio directly from
the labeled dataset text using the local fixture TTS engine — no upload, cloud
service, or download required. Each generated `AudioExample` carries the source
text as its reference transcript and a path to a WAV file.

The audio then runs through an audio tool-calling pipeline: **Pipeline C**
(direct audio in) for an audio-capable adapter such as Voxtral, or **Pipeline
D** (cascaded ASR → text) for text-only adapters like the mock, so this section
works even on the no-download dry run.

In [ ]:
import pandas as pd

# Generate test audio from the labeled bilingual dataset text (no upload needed).
audio_dataset = demo.demo_dataset()  # English + Russian
audio_examples = demo.synthesize_demo_audio(audio_dataset, output_dir="demo_audio")
print(f"synthesized {len(audio_examples)} audio clips (en + ru) into demo_audio/")
display(pd.DataFrame(demo.audio_summary(audio_examples)))

# Run the audio examples through the audio tool-calling pipeline. Pipeline C for
# audio-capable adapters (e.g. voxtral); Pipeline D (ASR -> text) otherwise.
audio_records = demo.run_audio_demo(
    adapter, audio_examples, run_id=f"colab-audio-{ADAPTER_ID}"
)
print("pipeline:", audio_records[0].pipeline)
display(pd.DataFrame(demo.record_summary(r) for r in audio_records))

# Audio metrics broken down by language (English vs Russian).
display(demo.metrics_by_language(audio_records, audio_dataset))

## 6. Validation, execution, and final answers

Inspect parse status, validation errors, executed tool results, and final
answers for each example.

In [ ]:
import pandas as pd

summary = pd.DataFrame(demo.record_summary(r) for r in records)
summary[["example_id", "parsable", "validation_errors", "tool", "final_answer"]]

## 7. Metrics for the selected adapter

Score the selected adapter on the labeled **bilingual** demo dataset (English
and Russian conversion prompts that need a tool, plus general-knowledge prompts
that do not). The labels let us measure tool decision accuracy, full tool-call
exact match, and the precision/recall trade-off — broken down per language so
the English/Russian gap is visible.

In [ ]:
dataset = demo.demo_dataset()  # bilingual: English + Russian
print(f"labeled demo examples: {len(dataset)} (languages: en, ru)")

scored = demo.run_examples(adapter, dataset, run_id=f"colab-metrics-{ADAPTER_ID}")
demo.metrics_by_language(scored, dataset)

## 8. Compare models

Run several adapters over the **same** bilingual dataset and compare their
metrics side by side, with a row per (model, language) plus an `all` row per
model. Keep `["mock"]` for a no-download dry run, or list the real adapters to
benchmark them against each other on English and Russian.

> **Heads up:** each real adapter downloads weights on first use and needs a
> GPU/Colab runtime. Running all three (`qwen`, `gemma`, `voxtral`) pulls tens
> of GB. An adapter that fails to run is reported in an `error` column instead
> of aborting the whole comparison.

In [ ]:
# Models to compare. Use ["mock"] for a no-download dry run, or e.g.
# ["qwen", "gemma", "voxtral"] to benchmark the real adapters on a GPU runtime.
COMPARE_ADAPTERS = ["mock"]

records_by_model, comparison = demo.compare_models(
    COMPARE_ADAPTERS, dataset, by_language=True
)
comparison

In [ ]:
# Rank the models that ran successfully by full tool-call exact match, the
# strictest end-to-end metric (correct decision + tool name + all arguments).
# Pivot the per-language rows so each language is a column for easy comparison.
ran = comparison[comparison["error"].isna()]
if not ran.empty:
    overall = ran[ran["language"] == "all"].sort_values(
        "tool_call_exact_match", ascending=False
    )
    display(overall[["model", "tool_decision_accuracy", "tool_call_exact_match"]])
    print("best on tool_call_exact_match (all):", overall.iloc[0]["model"])

    by_lang = ran.pivot(
        index="model", columns="language", values="tool_call_exact_match"
    )
    print("\ntool_call_exact_match by language:")
    display(by_lang)
else:
    print("No adapters ran successfully; see the 'error' column above.")

## 9. Compare every model across pipelines A–D

Run each model over the **same** data through all four pipelines and compare the
metrics side by side:

- **A** — text in → JSON tool-call envelope (tool metrics)
- **B** — audio in → ASR transcript (WER vs the reference transcript; an ASR
  baseline, model-independent in the current stack)
- **C** — audio in → direct audio tool-call envelope (tool metrics; needs an
  audio-capable model such as Voxtral)
- **D** — audio in → ASR transcript then text-LLM tool-call envelope (tool
  metrics **and** WER)

The audio is synthesized once so every model and pipeline sees identical inputs.
Metrics reuse the shared `summarize_metrics`, including a **modality gap** that
compares each audio pipeline with the Pipeline A text baseline. Pipelines a model
cannot serve (e.g. C for text-only adapters) appear in the skips table rather
than the metrics.

In [ ]:
# Run every model across pipelines A-D on the same data (uses COMPARE_ADAPTERS
# from section 8). Audio is synthesized once and shared across all runs.
records_by_model, pipeline_comparison, pipeline_skips = demo.compare_pipelines(
    COMPARE_ADAPTERS, dataset
)

# Headline numbers: overall across splits and languages, one row per (model, pipeline).
overall = pipeline_comparison[
    (pipeline_comparison["split"] == "all") & (pipeline_comparison["language"] == "all")
]
display(
    overall[
        [
            "model",
            "pipeline",
            "parsable_tool_invocation_rate",
            "tool_decision_accuracy",
            "tool_call_exact_match",
            "wer",
            "modality_gap",
        ]
    ].reset_index(drop=True)
)

if not pipeline_skips.empty:
    print("Skipped (model, pipeline) — adapter cannot serve the pipeline:")
    display(pipeline_skips)

In [ ]:
# Pivot the headline numbers into model x pipeline matrices for the two metrics
# that matter most: tool-call exact match (A/C/D) and WER (B/D).
if not overall.empty:
    exact = overall.pivot(
        index="model", columns="pipeline", values="tool_call_exact_match"
    )
    print("tool_call_exact_match by pipeline (tool pipelines A/C/D):")
    display(exact)

    wer = overall.pivot(index="model", columns="pipeline", values="wer")
    print("WER by pipeline (transcribing pipelines B/D):")
    display(wer)
else:
    print("No pipeline runs to compare; see the skips table above.")